# СИМА — Сервис рельефа (демо)

Демонстрация библиотеки **`sima-relief-service`** — сервисного слоя блока
«Анализ рельефа» из Q3-концепции. Библиотека пригодна для Jupyter-демо и
переиспользования; структурно готова к обёртке в реальный backend-сервис
(FastAPI/Celery/S3), но сама инфраструктуру не реализует.

Покрывает:
1. Оценка материалов ВЛС (LAS) и АФС (TIFF) — СК, площадь, разрешение, плотность, диапазон высот ТЛО.
2. Расчёт ЦМР (DTM) из LAS (SMRF) или использование существующей ЦМР.
3. Производные: уклон, экспозиция, TPI (.geotiff).
4. Векторные слои: горизонтали (.shp), отметки высот (.shp, las|dem), TIN (.dxf).
5. Сессионное хранение, статусы по тайлам, failed_tiles, детерминизм.

Структура и пути переиспользованы из `sima_dsm_demo.ipynb`.

In [ ]:
import sys, os, json
from pathlib import Path

backend = Path('/Users/sergeyzay/Documents/НЕДРА/СИМА/sima-web/backend')
for pkg in ['packages/sima-dem-core/src', 'packages/sima-dem-ground/src',
            'packages/sima-dem-dsm/src', 'packages/sima-dem-pipeline/src',
            'packages/sima-relief-service/src']:
    sys.path.insert(0, str(backend / pkg))

import rasterio, numpy as np
from sima_relief_service import (
    ReliefService, ReliefRequest, ReliefParams, TileInput,
    DerivativesParams, VectorsParams, HeightsParams, SmrfParams, SmoothingParams,
    assess_materials,
)
print('Импорт готов')

## 1. Датасет и пути

Переключатель `DATASET = 'demo' | 'test'` (как в `sima_dsm_demo.ipynb`).

In [ ]:
DATASET = 'demo'

if DATASET == 'demo':
    LAS_PATH = '/Users/sergeyzay/Documents/НЕДРА/СИМА/23_04_12_digital_elevation_1-46-315/demo_data/pt000100.las'
    TIF_PATH = '/Users/sergeyzay/Documents/НЕДРА/СИМА/23_04_12_digital_elevation_1-46-315/demo_data/00000100.tif'
    REF_DIR = Path(LAS_PATH).parent
elif DATASET == 'test':
    LAS_PATH = '/Users/sergeyzay/Documents/НЕДРА/СИМА/test_data/P-42-041-239-g_ground_TLO.las'
    TIF_PATH = '/Users/sergeyzay/Documents/НЕДРА/СИМА/test_data/P-42-041-239-g.tif'
    REF_DIR = Path(TIF_PATH).parent

OUTPUT_DIR = str(backend / 'output' / f'relief_service_notebook_{DATASET}')
os.makedirs(OUTPUT_DIR, exist_ok=True)

with rasterio.open(TIF_PATH) as src:
    CRS = src.crs.to_wkt()
print(f'Датасет: {DATASET}')
print(f'ВЛС: {LAS_PATH}')
print(f'CRS: {CRS[:70]}...')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

## 2. Оценка материалов (Q3 «Оценка файлов ВЛС/АФС»)

`assess_materials` извлекает: СК, площадь экстента, разрешение (АФС), плотность
точек и диапазон высот ТЛО (ВЛС). Масштаб ОФП/ТЛО — параметр съёмки (не из файла).

In [ ]:
assessment = assess_materials(vls_files=[LAS_PATH], afs_files=[TIF_PATH])
vls, afs = assessment.vls, assessment.afs
print('=== ВЛС (LAS) ===')
print(f'  СК задана: {bool(vls.crs)}  площадь, км²: {vls.extent_area_km2:.4f}')
print(f'  плотность, pts/m²: {vls.density_pts_m2:.2f}')
print(f'  диапазон высот ТЛО, м: {vls.tlo_height_range_m}')
print('=== АФС (TIFF) ===')
print(f'  разрешение, м: {afs.resolution_m:.3f}  площадь, км²: {afs.extent_area_km2:.4f}')
print(f'  тайлов: {afs.tiles_total}, ok: {afs.tiles_ok}, failed: {afs.tiles_failed}')

## 3. Запуск сервиса рельефа

Последовательность шагов по тайлам: crop → filter → ЦМР → smooth → derivatives → vectors → heights.
Трекаются статусы шагов; упавший тайл → failed + причина, остальные продолжаются.

In [ ]:
RESOLUTION = 1.0
params = ReliefParams(
    target_crs=CRS, filter_method='smrf',
    smrf=SmrfParams(slope=0.2, window=16, threshold=0.45, scalar=1.2),
    smoothing=SmoothingParams(enabled=True, sigma=1.0, order=0, window=5),
    derivatives=DerivativesParams(slopes=True, slopes_res=RESOLUTION,
                                  aspect=True, aspect_res=RESOLUTION,
                                  tpi=True, interpolation=True, inter_amp=100),
    vectors=VectorsParams(horizontals=[0.5, 2.0, 5.0], tin=True),
    heights=HeightsParams(enabled=True, source='dem', step=10),
    deterministic=True, seed=42,
)
request = ReliefRequest(
    params=params, project_id='demo_project', resolution=RESOLUTION,
    tiles=[TileInput(name='pt000100', vls_path=LAS_PATH, afs_path=TIF_PATH)],
)
svc = ReliefService(root_dir=OUTPUT_DIR)
result = svc.run(request)
job = result.job
print(f'Job: {job.status} | сессия: {job.session_id}')
print(f'Тайлов: всего {job.tiles_total}, done {job.tiles_done}, failed {job.tiles_failed}, прогресс {job.progress}%')

In [ ]:
tile = job.tiles[0]
print(f'Тайл {tile.name}: {tile.status} (id={tile.id})')
print('Шаги:')
for s in tile.steps:
    msg = f' — {s.message}' if s.message else ''
    print(f'  {s.name:10} {s.status:8}{msg}')
print('Артефакты (Q3 форматы):')
for a in tile.output_files:
    print(f'  [{a.kind:7}] {a.layer:12} {Path(a.path).name} ({a.size_bytes or 0} байт)')

## 4. Верификация против эталонных выходов `demo_data`

Для `DATASET='demo'` сравниваем ЦМР/уклон/экспозицию/TPI сервиса с эталонами
`pt000100_{dem,slope,aspect,tpi}.tif` (legacy-конвейер).

In [ ]:
def raster_stats(p):
    with rasterio.open(p) as src:
        a = src.read(1); m = src.read_masks(1)
        a = a[m > 0]
        return float(a.min()), float(a.max()), float(a.mean())
def find(layer):
    for a in tile.output_files:
        if a.layer == layer: return a.path
    return None
if DATASET == 'demo':
    pairs = [('dtm', REF_DIR / 'pt000100_dem.tif'),
             ('slope', REF_DIR / 'pt000100_slope.tif'),
             ('aspect', REF_DIR / 'pt000100_aspect.tif'),
             ('tpi', REF_DIR / 'pt000100_tpi.tif')]
    print(f'{"слой":8} {"сервис min/max/mean":28} {"эталон min/max/mean":28}')
    for layer, ref in pairs:
        got = find(layer)
        if got and Path(ref).exists():
            g, r = raster_stats(got), raster_stats(str(ref))
            print(f'{layer:8} {g[0]:7.2f}/{g[1]:7.2f}/{g[2]:7.2f}   {r[0]:7.2f}/{r[1]:7.2f}/{r[2]:7.2f}')
        else:
            print(f'{layer:8} — нет выходного или эталона')
else:
    print('Верификация против эталона доступна только для demo')

## 5. Повторный запуск тайла — история не затирается

Новый `tile_id` при повторе (REFINEMENT_PLAN Г1).

In [ ]:
result2 = svc.run(request)
t1, t2 = result.job.tiles[0], result2.job.tiles[0]
print(f'Запуск 1: id={t1.id}')
print(f'Запуск 2: id={t2.id}')
print(f'ID различаются (история не затёрта): {t1.id != t2.id}')
print(f'Старая сессия сохранена: {Path(t1.output_dir).exists()}')

## Итог
`sima-relief-service` — отдельная библиотека сервисного слоя рельефа: работает с LAS/TIFF,
извлекает метаданные материалов, корректно считает ЦМР/ЦМД и производные, выводит
форматы Q3 (.geotiff/.shp/.dxf), ведёт сессии и статусы по тайлам, готова к обёртке в
реальный backend-сервис (FastAPI/Celery/S3).